In [1]:
!pip install unsloth
!pip install datasets trl accelerate bitsandbytes

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.13/site-packages/triton/runtime/autotuner.py:101: DeprecationWarning: warmup, rep, and use_cuda_graph parameters are deprecated. See https://github.com/triton-lang/triton/pull/4496 for details.
  warnings.warn(("warmup, rep, and use_cuda_graph parameters are deprecated. See "


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/opt/conda/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=1808) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = [
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
)

Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/ykaitao/lingua_env_data/resolve/main/ambiguous_examples.jsonl"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'target_index', 'word', 'candidate_parses', 'gold'],
        num_rows: 10
    })
})


In [5]:
def reward_parse(prompts, completions, answer, **kwargs):

    rewards = []

    for completion, correct in zip(completions, answer):

        pred = completion.strip()

        if pred == correct:
            rewards.append(1.0)
        else:
            rewards.append(0.0)

    return rewards

In [8]:
print(dataset["train"].column_names)

['sentence', 'target_index', 'word', 'candidate_parses', 'gold']


In [9]:
def format_prompt(example):

    sentence = example["sentence"]
    parses = example["candidate_parses"]

    parse_text = ""
    for i,p in enumerate(parses):
        parse_text += f"{i}: {p}\n"

    prompt = f"""
You are a linguistics expert.

Sentence:
{sentence}

Candidate parses:
{parse_text}

Which parse correctly represents the sentence?

Respond with ONLY the index number.
"""

    return {
        "prompt": prompt,
        "answer": str(example["target_index"])
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [10]:
from trl import GRPOConfig

training_args = GRPOConfig(
    learning_rate = 5e-6,
    num_generations = 8,
    max_steps = 1000,
    per_device_train_batch_size = 1,
    logging_steps = 10,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


In [11]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    train_dataset = dataset,
    reward_funcs = [reward_parse],
)

In [12]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,000 | Num Epochs = 9,223,372,036,854,775,807 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)
There seems not to be a single sample in your epoch_iterator, stopping training at step 0! This is expected if you're using an IterableDataset and set num_steps (1000) higher than the number of available samples.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / reward_parse / mean,rewards / reward_parse / std


TrainOutput(global_step=0, training_loss=0.0, metrics={'train_runtime': 0.004, 'train_samples_per_second': 3989588.253, 'train_steps_per_second': 249349.266, 'total_flos': 0, 'train_loss': 0.0})

In [13]:
def choose_parse(sentence, parses):

    parse_text = ""
    for i,p in enumerate(parses):
        parse_text += f"{i}: {p}\n"

    prompt = f"""
Sentence:
{sentence}

Candidate parses:
{parse_text}

Respond with the correct index.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(**inputs, max_new_tokens=5)

    result = tokenizer.decode(output[0], skip_special_tokens=True)

    return int(result.strip())

In [15]:
from models import LinguaAction
from openenv.core.env_client import EnvClient

class LinguaClient(EnvClient[LinguaAction, dict, dict]):
    def _step_payload(self, action):
        return action.message

    def _parse_result(self, payload):
        return payload

    def _parse_state(self, payload):
        return payload

client = LinguaClient(base_url="http://127.0.0.1:8000")



In [16]:
from models import LinguaAction
from openenv.core.env_client import EnvClient

class LinguaClient(EnvClient[LinguaAction, dict, dict]):
    def _step_payload(self, action):
        return action.message

    def _parse_result(self, payload):
        return payload

    def _parse_state(self, payload):
        return payload


with LinguaClient(base_url="http://127.0.0.1:8000") as client:

    obs = client.reset()["observation"]

    sentence = obs["sentence"]
    parses = obs["candidate_parses"]

    print(sentence)
    print(parses)

['Начальник', 'областного', 'управления', 'связи', 'Семен', 'Еремеевич', 'был', 'человек', 'простой', ',', 'приходил', 'на', 'работу', 'всегда', 'вовремя', ',', 'здоровался', 'с', 'секретаршей', 'за', 'руку', 'и', 'иногда', 'даже', 'писал', 'в', 'стенгазету', 'заметки', 'под', 'псевдонимом', '"', 'Муха', '"', '.']
[{'lemma': 'писать', 'tag': 'VERB,impf,tran masc,sing,past,indc', 'score': 0.99115}, {'lemma': 'писать', 'tag': 'VERB,impf,intr masc,sing,past,indc', 'score': 0.008849}]
